# CCP estimation of the Zurcher model

Class 13 of ECO 629, notes at <https://dse.iskh.me/ccp-practice>.

## CCP estimator

**Stage 1** 

Using data pairs $(x_t,d_t)$ over a panel with at least two periods:

- estimate CCP $P(\text{replace}|x)$ for each $x$
- estimate transition probabilities $\Pi_{x,x'}$ when the engine is kept for each $x$ and $x'$

using consistent estimator, e.g. the frequency estimator or a flexible logit model

**Stage 2**

Form a (pseudo) statistical criterion as a function of first stage estimates and the structural parameters $(RC, c)$, and apply standard estimation technique, for example GMM, MSM or MLE.

In our case we have $P(\text{keep}|x)$ as a function of $(RC, c)$ and the first-stage CCPs, so we can use MLE in a standard 
**static binary logit** with a given offset. 

There is no fixed point to solve, not once!

### The NFXP benchmark and the offset

The representation is exact, not an approximation. Here we need the NFXP estimates as
the benchmark to compare with, and the model's own CCPs at those estimates for the plots
below.

In [ ]:
import sys
sys.path.insert(0, '../session10-11')        # zurcher.py, nfxp.py and the bus data

import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import chebyshev
from nfxp import read_busdata, estim_zurcher

In [ ]:
data = read_busdata()                        # bus groups 1-4, 175 bins, 450 000 miles
est = estim_zurcher(data)                    # the model with the data attached
res = est.estimate(full=False)               # NFXP partial likelihood: (RC, c), p fixed
print(res)

ev, pk, _ = est.solve()                      # the model at the NFXP estimates
_, v_rep, L = est.choice_values(ev)
logP_rep = v_rep - L                         # log P(replace|x) as a difference, never log of a probability

In [ ]:
def fd_offset(model, logP_rep):
    '''Future term of v(x,keep) - v(x,replace) under one-period finite dependence:
    -beta * sum_x' log P(replace|x') [pi(x'|x,keep) - pi(x'|x,replace)]'''
    Pi = model.trpr                                  # keep: row x of the transition matrix
    return -model.beta * (Pi - Pi[0]) @ logP_rep     # replace: row 0, a fresh engine, for every x

## First stage: CCPs from the data

The natural first stage is the bin estimator, replacements over visits in each mileage
bin:

In [ ]:
def ccp_frequency(model):
    '''Bin estimator of P(replace|x): replacements over visits, NaN where never visited'''
    visits = np.bincount(model.x, minlength=model.n)
    replacements = np.bincount(model.x, weights=model.d, minlength=model.n)
    with np.errstate(invalid='ignore'):              # 0/0 in the unvisited bins
        return replacements / visits, visits

P_freq, visits = ccp_frequency(est)
print(f'{(visits == 0).sum()} of {est.n} bins never visited, '
      f'{((visits > 0) & (P_freq == 0)).sum()} visited but no replacement observed')
with np.errstate(divide='ignore'):
    offset = fd_offset(est, np.log(P_freq))
print(f'finite offsets: {np.isfinite(offset).sum()} of {est.n}')

In [ ]:
def logit_mle(X, y, offset=0.0, tol=1e-10, maxiter=100, callback=None):
    '''Binary logit P(y=1|X) = Lambda(X b + offset) by Newton's method with step halving,
    with given tolerance and number of iterations.
    Callback function is invoked at each iteration if given.
    Returns b, standard errors from the inverse Hessian, and the log-likelihood.
    '''
    def loglik(b):
        z = X @ b + offset
        return np.sum(y * z - np.logaddexp(0, z))  # log Lambda(z) = z - log(1 + e^z), no overflow
    b0 = np.zeros(X.shape[1])
    for i in range(maxiter):
        p = 0.5 * (1 + np.tanh((X @ b0 + offset) / 2))  # Lambda(z), stable for any z
        g = X.T @ (y - p)                              # score
        H = (X * (p * (1 - p))[:, None]).T @ X         # minus the Hessian, positive definite
        step = np.linalg.solve(H, g)                   # Newton step
        while loglik(b0 + step) < loglik(b0) and np.abs(step).max() > tol:
            step /= 2                                  # plain Newton overshoots from b = 0
        b1 = b0 + step
        err = np.abs(b1 - b0).max()
        if callback != None: callback(err=err, b0=b0, b1=b1, iter=i)
        if err < tol: break
        b0 = b1
    else:
        raise RuntimeError('Failed to converge in %d iterations' % maxiter)
    return b1, np.sqrt(np.diag(np.linalg.inv(H))), loglik(b1)

The structure is that of `newton()` in the
[Classic solvers lecture](https://dse.iskh.me/solvers): `tol`, `maxiter` and an optional `callback`,
and a `for ... else` loop that raises an error if the iterations run out.

The logit likelihood is globally concave, but that does not make plain Newton safe: from
$b=0$ the first step on the second-stage problem below lands at $c \approx -230$, the
next at $c \approx 120$, and the iterations diverge. Halving the step until the
likelihood improves fixes it. The first stage uses Chebyshev polynomials of mileage
mapped to $[-1,1]$, because with the plain powers $x^k$ the Hessian of the first stage
is numerically singular already at degree 5:

In [ ]:
def ccp_logit(model, degree=3, full=False):
    '''Flexible logit first stage: log P(replace|x) on the whole grid from a logit in
    Chebyshev polynomials of mileage, which extrapolates to the bins without replacements.
    With full=True also returns the coefficients, their standard errors and the log-likelihood.'''
    basis = lambda x: chebyshev.chebvander(2 * x / model.n - 1, degree)   # grid mapped to [-1, 1]
    b, se, ll = logit_mle(basis(model.x), model.d)
    logP = -np.logaddexp(0, -basis(model.grid) @ b)       # log Lambda(z) = -log(1 + e^{-z})
    return (logP, b, se, ll) if full else logP

degrees = (1, 2, 3, 4, 5, 10)
first_stage, fits = {}, {}
for degree in degrees:
    logP, b, se, ll = ccp_logit(est, degree, full=True)
    first_stage[degree], fits[degree] = logP, (b, se, ll)

# table: one column per degree, estimates with standard errors in brackets
print(f'{"":<18}' + ''.join(f'{"degree " + str(d):>18}' for d in degrees))
for k in range(max(degrees) + 1):
    cells = [f'{b[k]:.3f} ({se[k]:.3f})' if k < b.size else '' for b, se, _ in fits.values()]
    print(f'{"T" + str(k):<18}' + ''.join(f'{c:>18}' for c in cells))
print(f'{"log-likelihood":<18}' + ''.join(f'{ll:>18.2f}' for _, _, ll in fits.values()))
print(f'{"log P(replace|0)":<18}' + ''.join(f'{first_stage[d][0]:>18.2f}' for d in degrees))

## Second stage: a static logit with an offset

With the offset computed from the first stage, the second stage is the logit of
[`logit_mle`] with two regressors, the constant for $RC$ and
$-0.001\,x$ for $c$, and the offset passed through:

In [ ]:
def estimate_ccp(model, logP_rep):
    '''Second stage: P(keep|x) = Lambda(RC - 0.001 c x + offset(x)), a static logit in
    (RC, c) with the future term computed from the first-stage CCPs'''
    offset = fd_offset(model, logP_rep)
    X = np.column_stack([np.ones(model.N), -0.001 * model.x])   # coefficients: RC and c
    return logit_mle(X, 1 - model.d, offset[model.x])           # y = 1 for keep

print(f'{"first stage":<16}{"RC":>8}{"s.e.":>8}{"c":>8}{"s.e.":>8}{"loglik":>10}{"-log P(rep|0)":>15}')
print(f'{"NFXP":<16}{res.theta[0]:8.3f}{res.se[0]:8.3f}{res.theta[1]:8.3f}{res.se[1]:8.3f}'
      f'{res.stages[0]["loglik_choice"]:10.2f}{-logP_rep[0]:15.2f}')
for degree, lp in first_stage.items():
    try:
        th, se, ll = estimate_ccp(est, lp)
        print(f'{"logit, degree " + str(degree):<16}{th[0]:8.3f}{se[0]:8.3f}{th[1]:8.3f}{se[1]:8.3f}'
              f'{ll:10.2f}{-lp[0]:15.2f}')
    except np.linalg.LinAlgError:                # predicted probabilities all 0 or 1
        print(f'{"logit, degree " + str(degree):<16}{"fails: singular Hessian":>42}{-lp[0]:15.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
miles = est.thousand_miles(est.grid)
ax.plot(miles, logP_rep, 'k', lw=2.5, label='NFXP model')
for degree, lp in first_stage.items():
    ax.plot(miles, lp, lw=1.2, label=f'first-stage logit, degree {degree}')
seen = P_freq > 0
ax.scatter(miles[seen], np.log(P_freq[seen]), s=14, c='gray', label='frequency, where positive')
ax.set(xlabel='mileage since last replacement, thousand miles', ylabel='log P(replace | x)',
       ylim=(-20, 0.5))
ax.legend(loc='lower right')
plt.show()

### The same second stage with estimagic

Writing the Newton iterations by hand shows what happens inside. In practice a package
does the optimization and the inference:

- [estimagic](https://estimagic.org/en/stable/) estimates a model by maximum likelihood
  given only the log-likelihood **contributions**, one per observation
- it is installed as part of the package `optimagic`, the optimization library it was
  renamed into; `import estimagic` still provides the estimation functions
- the optimizer is chosen by name, here L-BFGS-B from SciPy
- standard errors come from the outer product of the scores (`"jacobian"`, the default),
  the Hessian, or the robust sandwich of the two, all by numerical differentiation unless
  derivatives are supplied

The second version of `logit_mle` has the same inputs and outputs as the first:

In [ ]:
import optimagic as om                         # pip/conda package optimagic, which ships
import estimagic as em                         # estimagic alongside it as a separate module

@om.mark.likelihood                            # tells estimagic these are per-observation contributions
def logit_loglike(params, X, y, offset):
    '''Log-likelihood contribution of every observation of the binary logit'''
    z = X @ params + offset
    return y * z - np.logaddexp(0, z)

def logit_mle_em(X, y, offset=0.0, algorithm='scipy_lbfgsb'):
    '''Binary logit by estimagic: b, standard errors from the Hessian, log-likelihood'''
    res = em.estimate_ml(loglike=logit_loglike, params=np.zeros(X.shape[1]),
                         optimize_options=algorithm,
                         loglike_kwargs=dict(X=X, y=y, offset=offset))
    b = res.params
    return b, res.se(method='hessian'), logit_loglike(b, X, y, offset).sum()

In [ ]:
offset = fd_offset(est, first_stage[3])
X = np.column_stack([np.ones(est.N), -0.001 * est.x])
for name, fit in (('Newton', logit_mle), ('estimagic', logit_mle_em)):
    th, se, ll = fit(X, 1 - est.d, offset[est.x])
    print(f'{name:<10} RC {th[0]:8.4f} ({se[0]:.4f})  c {th[1]:7.4f} ({se[1]:.4f})  loglik {ll:9.4f}')

## Exercises


All four start from the objects defined above: `data`, `est`, `res`, `logP_rep`,
`P_freq` and the functions `fd_offset`, `logit_mle`, `ccp_logit` and `estimate_ccp`.

### Frequency first stage with a floor

The bin estimator becomes usable once the zeros are replaced by a small floor
$\varepsilon$, and so do the unvisited bins:

In [ ]:
for eps in (1e-2, 1e-3, 1e-4, 1e-6):
    lp = np.log(np.clip(np.nan_to_num(P_freq), eps, 1))   # NaN -> 0 -> eps
    th, se, ll = estimate_ccp(est, lp)
    print(f'floor {eps:g}:  RC {th[0]:7.3f}  c {th[1]:7.3f}  loglik {ll:9.2f}')

**Discussion**


- How do $RC$ and $c$ move with the floor, and why does $c$ move so much more than with
  the logit first stage?
- Is there a principled way to choose $\varepsilon$?

### Monte Carlo: which part of the error comes from the first stage?

Simulate panels from the model at the NFXP estimates, using `sim_zurcher.py` in the
folder of this class, and estimate each of them three ways: the **infeasible** two-step
estimator with the true CCPs `logP_rep` in the first stage, the feasible one with the
flexible logit, and NFXP.

In [ ]:
from sim_zurcher import simulate

for N in (100, 1000, 10000):                                # buses, 120 months each
    sim = simulate(est, N=N, T=120, seed=1, pk=pk, x0=0)    # new engines at the start, like the data
    m = estim_zurcher(sim, J=est.J, p=est.p)                # same grid and transition probabilities
    th_inf, se_inf, _ = estimate_ccp(m, logP_rep)           # true CCPs
    th_3, _, _ = estimate_ccp(m, ccp_logit(m, 3))           # estimated CCPs
    r = m.estimate(full=False)
    print(f'N {N:6d}  infeasible RC {th_inf[0]:6.3f} c {th_inf[1]:6.3f} (s.e. {se_inf[1]:.2f})  '
          f'logit-3 RC {th_3[0]:6.3f} c {th_3[1]:6.3f}  NFXP RC {r.theta[0]:6.3f} c {r.theta[1]:6.3f} (s.e. {r.se[1]:.2f})')

**Discussion**


- Is the infeasible estimator consistent? Is the feasible one, with the degree held at
  3 as $N$ grows?
- Compare the standard errors of $c$ of the infeasible estimator and NFXP. How many more
  buses does the two-step estimator need for the same precision?

### Estimation under different $\beta$

In [ ]:
for beta in (0.0, 0.9, 0.99, 0.9999):
    m = estim_zurcher(data, beta=beta)
    th, _, _ = estimate_ccp(m, ccp_logit(m, 3))
    r = m.estimate(full=False)
    print(f'beta {beta:<7} CCP RC {th[0]:8.3f} c {th[1]:7.3f}   NFXP RC {r.theta[0]:8.3f} c {r.theta[1]:7.3f}')

**Discussion**


- Why do the two estimators agree exactly at $\beta = 0$?
- $\beta$ enters the second stage as a coefficient on the offset. Why can it not be
  estimated as a third coefficient of the same logit?

### Run time

In [ ]:
%timeit estimate_ccp(est, ccp_logit(est, 3))
%timeit estim_zurcher(data).estimate(full=False)

Compare the two, then repeat on a grid of twice and four times as many bins,
`read_busdata(n=350, max_miles=900_000)` and so on, and report how each time scales with
$n$.